# Lab 09 · Certify roots and account for the search

**Preparation:** [Lecture 09](../notebooks/09_validated_roots.ipynb).

**Time:** 2 hours plus 30–60 minutes homework. Complete the local interval Newton decision routine, then use supplied search scaffolding to maintain coverage of the possible root set.

## Google Colab setup — run this cell first

**Local Jupyter:** this cell skips Colab setup; use your installed course environment.

Use a **CPU Python runtime** (Python 3.12 or newer). Run the next cell and select
**vc_runtime.zip** from this edition when prompted. It loads the course helpers
and installs the arithmetic libraries. Repeat setup whenever you get a new runtime.
No Drive mounting or local Python installation is required.
Use this setup instead of the local installation and kernel-selection instructions
in the original course text.

Save a copy of this notebook in Drive, or download it after editing. Files created
by the project are under `/content/validated-course/build/certificates/`; download
those separately from Colab's Files panel. Runtime files are temporary.

Open other course notebooks using **File → Upload notebook**, choosing them from
the extracted course folder. Relative course links are intended for local Jupyter
and do not open sibling notebooks automatically in Colab. In labs, complete exercise
cells before running their diagnostics: `NotImplementedError` marks an unfinished task.


In [1]:
try:
    import google.colab
except ImportError:
    print("Local Jupyter: using the installed course environment.")
else:
    import hashlib
    from pathlib import Path
    import subprocess
    import sys
    from google.colab import files

    if sys.version_info < (3, 12):
        raise RuntimeError("This course needs a Python 3.12 or newer Colab runtime.")

    # Upload the small companion file supplied with the Colab course edition.
    expected_hash = '833652c4a440f1c3a22bc6e68c59d533b41e6f9f1ff0b4af226f8fbee2244c1a'
    runtime_zip = Path("/content") / ("vc-course-" + expected_hash + ".zip")
    if not runtime_zip.exists():
        print("Select vc_runtime.zip from the extracted Colab course folder.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload only vc_runtime.zip, then run this cell again.")
        archive_bytes = next(iter(uploaded.values()))
        if hashlib.sha256(archive_bytes).hexdigest() != expected_hash:
            raise ValueError("Wrong vc_runtime.zip: use the one supplied with this notebook.")
        runtime_zip.write_bytes(archive_bytes)

    if hashlib.sha256(runtime_zip.read_bytes()).hexdigest() != expected_hash:
        raise ValueError("The cached vc archive has changed; start a fresh runtime.")

    # Keep Colab's existing NumPy and plotting stack when they satisfy these bounds.
    # Install MPFR/Arb bindings, not the full local Jupyter environment.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        'numpy>=1.26', 'matplotlib>=3.8', 'gmpy2==2.3.1', 'python-flint==0.9.0', 'jupytext>=1.16'
    ])
    import gmpy2
    import flint
    if gmpy2.version() != "2.3.1" or flint.__version__ != "0.9.0":
        raise RuntimeError("Old arithmetic libraries are still loaded. Restart the session and run setup first.")

    if str(runtime_zip) not in sys.path:
        sys.path.insert(0, str(runtime_zip))

    import vc
    if not vc.__file__.startswith(str(runtime_zip) + "/"):
        raise RuntimeError("Another vc copy is already loaded. Start a fresh runtime.")

    # Preserve the relative output paths used by the project notebooks.
    import os
    working_folder = Path("/content/validated-course") / 'labs'
    working_folder.mkdir(parents=True, exist_ok=True)
    os.chdir(working_folder)
    print("Course helpers ready:", vc.__version__)

Local Jupyter: using the installed course environment.


In [2]:
from fractions import Fraction
import inspect
from vc.intervals import RationalInterval as Interval
from vc.autodiff import Dual
from vc.roots import RootDecision, isolate_roots

def cubic(x):
    return (x+2)*(x-1)*(x-3)

def derivative(X):
    return cubic(Dual(X, Interval(1))).derivative

## Task A · Implement the local decision (50 minutes)

Implement these decisions in order: exclude if the function image misses zero; request a split if the derivative includes zero; compute interval Newton; exclude on empty intersection; certify on strict interior inclusion; contract if the retained width is less than half the input width; otherwise request a split. Return a `RootDecision` with the supporting quantities. Match the status strings in the starter.

In [ ]:
def student_inspect(evaluate, derivative, domain):
    # Statuses: excluded_range, excluded_newton, certified, contract, split.
    # RootDecision(domain, image, status, slopes=None, newton=None, retained=None)
    raise NotImplementedError("Follow the stated decision order")

In [ ]:
sqrt_two = student_inspect(lambda X: X.square()-2, lambda X: 2*X, Interval(1, 2))
assert sqrt_two.status == "certified"
assert sqrt_two.retained.lo**2 < 2 < sqrt_two.retained.hi**2
excluded = student_inspect(lambda X: X.square()+1, lambda X: 2*X, Interval(-1, 1))
assert excluded.status == "excluded_range"
multiple = student_inspect(lambda X: X.square(), lambda X: 2*X, Interval(-1, 1))
assert multiple.status == "split"
print("sqrt(2) certificate:", sqrt_two)

**Written response:** Prove root retention under intersection. Explain why dividing by a derivative interval containing zero is not a permitted fallback.

## Task B · Use and audit the search (45 minutes)

Run the supplied driver with your decision callback. Inspect the driver source and identify what happens to every box after exclusion, certification, contraction, splitting, or budget exhaustion. Check the known cubic roots independently.

In [ ]:
print(inspect.getsource(isolate_roots))
search = isolate_roots(cubic, derivative, Interval(-3, 4), inspect=student_inspect)
print(search.status)
for certificate in search.certified:
    print(certificate.domain, "->", certificate.retained)
print("Unresolved:", search.unresolved)

In [ ]:
assert search.status == "complete"
assert len(search.certified) == 3 and not search.unresolved
for certificate, root in zip(search.certified, [-2, 1, 3]):
    assert certificate.retained.contains(root)
for first, second in zip(search.certified, search.certified[1:]):
    assert first.retained.hi < second.retained.lo
for decision in search.decisions:
    for root in [-2, 1, 3]:
        if decision.domain.contains(root):
            assert not decision.status.startswith("excluded")
            if decision.retained is not None:
                assert decision.retained.contains(root)

**Written response:** State why this run establishes exactly three roots on the original domain. What does the tolerance control, and what does it not control?

## Task C · Report limitations honestly (35 minutes plus homework)

Run a small budget, a double root, and a boundary root. For each case list certified and unresolved regions. Do not convert an unresolved interval into a root merely because it is small.

In [ ]:
limited = isolate_roots(cubic, derivative, Interval(-3, 4), max_steps=2, inspect=student_inspect)
double = isolate_roots(lambda X: X.square(), lambda X: 2*X, Interval(-1, 1),
                       tolerance="1/100", inspect=student_inspect)
boundary = isolate_roots(lambda X: X, lambda X: Interval(1), Interval(0, 1),
                         inspect=student_inspect)
for name, result in [("budget", limited), ("double", double), ("boundary", boundary)]:
    print(name, result.status, "unresolved:", result.unresolved)
    assert result.unresolved
for root in [-2, 1, 3]:
    possible = limited.unresolved + [item.retained for item in limited.certified]
    assert any(part.contains(root) for part in possible)
assert any(part.contains(0) for part in double.unresolved)
assert any(part.contains(0) for part in boundary.unresolved)

**Written response:** Write a corrected version of “all roots were found to tolerance” for each run. As homework, explain how exact endpoint evaluation might help the boundary case and why deduplication would then matter.

## Submission

Complete the code and written responses. Restart the kernel and run every cell. State the mathematical assumptions for each guarantee; retain failed or inconclusive examples and explain them.